# Continuous causal response

This notebook estimates four related—but distinct—continuous-treatment estimands: a mean response curve, a local derivative, an elasticity, and an observed-law average derivative. It also shows why structural identification, empirical support, and uncertainty must be read separately.

The data are deterministic given the seed. Run this notebook from a source checkout after building the Python extension (`cd python && maturin develop --release`). Colab users can `pip install antecedent==0.7.0`.

In [ ]:
import numpy as np

import antecedent
from antecedent import (
    AverageDerivative,
    Elasticity,
    PointDerivative,
    ResponseCurve,
    analyze,
)

SEED = 500
N = 1_500

## A confounded nonlinear dose process

Baseline severity affects both the dose selected by clinicians and the outcome. The graph therefore requires adjustment for `severity`. Dose is positive, which gives the elasticity a meaningful treatment scale.

In [ ]:
rng = np.random.default_rng(SEED)
severity = rng.normal(size=N)
dose = np.clip(2.5 + 0.65 * severity + rng.normal(scale=0.65, size=N), 0.25, 5.0)
outcome = (
    12.0
    + 3.0 * np.log(dose)
    - 0.30 * dose**2
    + 1.25 * severity
    + rng.normal(scale=0.45, size=N)
)

data = {"severity": severity, "dose": dose, "outcome": outcome}
graph = [
    ("severity", "dose"),
    ("severity", "outcome"),
    ("dose", "outcome"),
]

float(dose.min()), float(np.median(dose)), float(dose.max())

## Ask four explicit questions

Scalar queries follow Antecedent's established `(treatment, outcome)` positional order. The curve deliberately includes `5.25`, beyond the clipped empirical maximum of `5.0`, so the support report has to preserve and flag the scientific request rather than silently trimming it.

In [ ]:
curve_query = ResponseCurve(
    "dose",
    "outcome",
    grid=[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.25],
)
point_query = PointDerivative("dose", "outcome", at=2.5)
elasticity_query = Elasticity("dose", "outcome", at=2.5)
average_query = AverageDerivative("dose", "outcome", weighting="observed")

curve = analyze(data, graph=graph, query=curve_query)
point = analyze(data, graph=graph, query=point_query)
elasticity = analyze(data, graph=graph, query=elasticity_query)
average = analyze(data, graph=graph, query=average_query)

## Inspect the orthogonal result axes

An identified estimand can have weak or absent empirical support. Likewise, `pointwise` uncertainty describes coverage at individual grid points; it is not a simultaneous band for the whole curve. The provenance operation identifies the implemented algorithm, while assumptions remain inspectable rather than implicit.

In [ ]:
def result_summary(result):
    return {
        "identification": result.identification,
        "support": result.support.status,
        "support_warnings": list(result.support.warnings),
        "uncertainty": result.uncertainty.kind,
        "assumptions": list(result.assumptions),
        "provenance": dict(result.provenance),
    }

result_summary(curve)

In [ ]:
curve_rows = [
    {"dose": point_values[0], "mean_response": response_values[0]}
    for point_values, response_values in zip(
        curve.response.points, curve.response.values, strict=True
    )
]
curve_rows

In [ ]:
{
    "local_derivative_at_2_5": point.estimate,
    "elasticity_at_2_5": elasticity.estimate,
    "observed_law_average_derivative": average.estimate,
    "local_support": point.support.status,
    "average_support": average.support.status,
}

## Interpretation

The curve describes levels under intervention. The point derivative is the local slope at dose 2.5. The elasticity rescales a local derivative and is not a separate robustness check. The average derivative instead integrates slopes over the observed treatment law, so it answers a population-weighted question.

Do not report the final curve point as empirically supported merely because an estimator returned a number. Inspect `curve.support`, its diagnostics, and its warnings. Also do not describe pointwise intervals as a simultaneous confidence band.